# Embedding-verandering: betekenisverschil in plaats van pixelverschil

Een voorgetraind beeldnetwerk (ResNet18, getraind op miljoenen foto's) vat elke
pand-uitsnede samen in een **embedding**: een vector van 512 getallen die beschrijft
*wat er te zien is* — dakvorm, textuur, objecten — in plaats van welke pixel welke
waarde heeft. De afstand tussen de 2022-vector en de 2025-vector van hetzelfde pand
is dan een verandermaat die van nature ongevoelig is voor belichting, JPEG-ruis en
kleine verschuivingen: die veranderen de pixels, maar nauwelijks de betekenis.

Dit notebook is een **A/B-test**: hij berekent in één run zowel de pixel-verschilscore
(het v2-recept: contourmasker, verschuivings-tolerant) als de embedding-afstand, en
vergelijkt ze op de vaste meetlat (nieuwbouw 2023–2024), de werkvoorraad en de
meningsverschillen.

Vereist: `el_2022` + `el_2025` (stap 6), `pip install -r requirements-train.txt`
(voor torch), en eenmalig internet voor de modelgewichten (~45 MB, wordt gecachet).
Reken op 30–60 min CPU voor de hele stad.

In [ ]:
import io, json, sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from shapely.geometry import shape

try:
    import torch
    from torchvision.models import resnet18, ResNet18_Weights
except ImportError:
    raise SystemExit('torch/torchvision ontbreken — draai: pip install -r requirements-train.txt')

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import laad_config
from pdok import fetch_bag_panden, make_session, wms_get_map

cfg = laad_config(REPO / 'config.yaml')
DATA = REPO / cfg['paden']['data']
RES = cfg['luchtfoto']['resolutie']
PX = cfg['luchtfoto']['tegelgrootte']
STAP = PX * RES

GEBIED = [101500, 395000, 107000, 400000]
OUD_LAAG, NIEUW_LAAG = '2022_orthoHR', '2025_orthoHR'
OUD_MAP, NIEUW_MAP = DATA / 'el_2022', DATA / 'el_2025'
BAG_ALLE = DATA / 'bag' / 'el_panden_alle.geojson'
DREMPEL_PCT = 95
STEDELIJK_VANAF = 6
CROP_MARGE_M = 3       # meters context rond de pand-box in de uitsnede
BATCH = 64
MARKEER = '#FFD400'
BLAUW, GRIJS, AMBER = '#2563EB', '#94A3B8', '#F59E0B'
sessie = make_session()

for m in (OUD_MAP, NIEUW_MAP):
    assert (m / 'tiles.json').exists(), f'{m} ontbreekt — draai stap 6'
idx_oud = json.loads((OUD_MAP / 'tiles.json').read_text())
idx_nieuw = json.loads((NIEUW_MAP / 'tiles.json').read_text())
if not BAG_ALLE.exists():
    feats = fetch_bag_panden(sessie, cfg['bag']['wfs_url'], tuple(GEBIED), alleen_in_gebruik=False)
    BAG_ALLE.write_text(json.dumps({'type': 'FeatureCollection', 'bbox_rd': GEBIED, 'features': feats}))
panden = json.loads(BAG_ALLE.read_text())['features']
nieuwbouw = [f for f in panden if 2022 <= (f['properties'].get('bouwjaar') or 0) <= 2025]
bag_verklaard_ids = {f['properties']['identificatie'] for f in panden
                     if (f['properties'].get('bouwjaar') or 0) >= 2022
                     or f['properties'].get('status') in
                     ('Verbouwing pand', 'Sloopvergunning verleend',
                      'Bouwvergunning verleend', 'Bouw gestart')}
print(f'{len(panden)} BAG-panden, {len(set(idx_oud) & set(idx_nieuw))} tegelparen geladen')

## 1. Het embedding-model

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Identity()      # classificatiekop eraf: we willen de 512-d vector
model.eval()
IMAGENET_GEM = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def naar_tensor(crop):
    t = torch.from_numpy(np.asarray(crop.resize((224, 224)), dtype=np.float32) / 255.0)
    return (t.permute(2, 0, 1) - IMAGENET_GEM) / IMAGENET_STD

@torch.no_grad()
def embed(crops):
    e = model(torch.stack([naar_tensor(c) for c in crops]))
    e = e / e.norm(dim=1, keepdim=True)
    return e.numpy()

print('Model klaar (ResNet18, 512-d embeddings, genormaliseerd).')

## 2. Beide scores in één doorloop

Per pand: (a) de pixel-verschilscore (v2-recept) en (b) de embedding-afstand
(1 − cosinusgelijkenis tussen de jaarvectoren, 0 = identiek, hoger = meer veranderd).

In [ ]:
GRID_X, GRID_Y = GEBIED[0], GEBIED[1]
def tegel_id_voor(x, y):
    return f't_{int((x - GRID_X) // STAP):04d}_{int((y - GRID_Y) // STAP):04d}'

def laad_genorm(pad):
    beeld = Image.open(pad).convert('L').filter(ImageFilter.GaussianBlur(1.5))
    a = np.asarray(beeld, dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

per_tegel = defaultdict(list)
for f in panden:
    geom = shape(f['geometry'])
    if geom.area < 25:
        continue
    per_tegel[tegel_id_voor(geom.centroid.x, geom.centroid.y)].append((f, geom))

paren_tegels = [tid for tid in per_tegel if tid in idx_oud and tid in idx_nieuw]
VERSCHUIVINGEN = [(dx, dy) for dx in (-8, 0, 8) for dy in (-8, 0, 8)]
MARGE_PX = int(CROP_MARGE_M / RES)

resultaten = {}       # pid -> dict(pixel=…, stratum=…, f=…, geom=…)
buffer, buffer_pids = [], []
embeddings = {}       # pid -> (e_oud, e_nieuw)

def spoel_buffer():
    if not buffer:
        return
    e = embed([c for paar in buffer for c in paar])
    for k, pid in enumerate(buffer_pids):
        embeddings[pid] = (e[2 * k], e[2 * k + 1])
    buffer.clear(); buffer_pids.clear()

from tqdm.auto import tqdm
for tid in tqdm(paren_tegels, desc='tegels scoren'):
    rgb_oud = Image.open(OUD_MAP / idx_oud[tid]['image']).convert('RGB')
    rgb_nieuw = Image.open(NIEUW_MAP / idx_nieuw[tid]['image']).convert('RGB')
    a = laad_genorm(OUD_MAP / idx_oud[tid]['image'])
    b = laad_genorm(NIEUW_MAP / idx_nieuw[tid]['image'])
    minverschil = np.full_like(a, np.inf)
    for dx, dy in VERSCHUIVINGEN:
        np.minimum(minverschil, np.abs(np.roll(b, (dy, dx), axis=(0, 1)) - a), out=minverschil)
    bbox = idx_oud[tid]['bbox']
    stratum = ('stedelijk' if idx_oud[tid].get('n_panden', 0) >= STEDELIJK_VANAF
               else 'buitengebied')
    for f, geom in per_tegel[tid]:
        gx0, gy0, gx1, gy1 = geom.bounds
        x0 = max(0, int((gx0 - bbox[0]) / RES)); x1 = min(PX, int((gx1 - bbox[0]) / RES))
        y0 = max(0, int((bbox[3] - gy1) / RES)); y1 = min(PX, int((bbox[3] - gy0) / RES))
        if x1 - x0 < 12 or y1 - y0 < 12:
            continue
        masker = Image.new('1', (x1 - x0, y1 - y0), 0)
        tekenaar = ImageDraw.Draw(masker)
        for poly in (geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]):
            punten = [((px - bbox[0]) / RES - x0, (bbox[3] - py) / RES - y0)
                      for px, py in poly.exterior.coords]
            tekenaar.polygon(punten, fill=1)
        binnen = np.asarray(masker, dtype=bool)
        venster = minverschil[y0:y1, x0:x1]
        pixel = float(venster[binnen].mean()) if binnen.sum() >= 100 else float(venster.mean())
        pid = f['properties']['identificatie']
        resultaten[pid] = {'pixel': pixel, 'stratum': stratum, 'f': f, 'geom': geom}
        cx0 = max(0, x0 - MARGE_PX); cy0 = max(0, y0 - MARGE_PX)
        cx1 = min(PX, x1 + MARGE_PX); cy1 = min(PX, y1 + MARGE_PX)
        buffer.append((rgb_oud.crop((cx0, cy0, cx1, cy1)),
                       rgb_nieuw.crop((cx0, cy0, cx1, cy1))))
        buffer_pids.append(pid)
        if len(buffer) >= BATCH:
            spoel_buffer()
spoel_buffer()

for pid, (e1, e2) in embeddings.items():
    resultaten[pid]['embedding'] = float(1.0 - np.dot(e1, e2))
resultaten = {pid: r for pid, r in resultaten.items() if 'embedding' in r}
print(f'{len(resultaten)} panden dubbel gescoord (pixel + embedding)')

## 3. A/B: recall, werkvoorraad en samenhang

In [ ]:
pix = np.array([r['pixel'] for r in resultaten.values()])
emb = np.array([r['embedding'] for r in resultaten.values()])
strata = np.array([r['stratum'] for r in resultaten.values()])
pids = list(resultaten)

def detecties_van(waarden):
    drempels = {n: float(np.percentile(waarden[strata == n], DREMPEL_PCT))
                for n in ('stedelijk', 'buitengebied') if (strata == n).any()}
    return {pid for pid, w, s in zip(pids, waarden, strata) if w >= drempels[s]}

det_pix = detecties_van(pix)
det_emb = detecties_van(emb)

meetlat = [f['properties']['identificatie'] for f in nieuwbouw
           if 2023 <= f['properties']['bouwjaar'] <= 2024]
meetlat = [i for i in meetlat if i in resultaten]
recall_pix = sum(1 for i in meetlat if i in det_pix) / max(1, len(meetlat))
recall_emb = sum(1 for i in meetlat if i in det_emb) / max(1, len(meetlat))

def buiten_aandeel(ids):
    return (sum(1 for i in ids if resultaten[i]['stratum'] == 'buitengebied') / len(ids)) if ids else 0

print(f'Meetlat: {len(meetlat)} nieuwbouwpanden (2023-2024)')
print(f'pixel:     {len(det_pix)} detecties, recall {recall_pix:.0%}, '
      f'{buiten_aandeel(det_pix):.0%} buitengebied')
print(f'embedding: {len(det_emb)} detecties, recall {recall_emb:.0%}, '
      f'{buiten_aandeel(det_emb):.0%} buitengebied')
print(f'eens over {len(det_pix & det_emb)}; alleen-pixel {len(det_pix - det_emb)}, '
      f'alleen-embedding {len(det_emb - det_pix)}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
ax1.bar(['pixel\n(v2)', 'embedding'], [recall_pix * 100, recall_emb * 100],
        color=[GRIJS, BLAUW], width=0.5)
for i, r_ in enumerate((recall_pix, recall_emb)):
    ax1.text(i, r_ * 100 + 1.5, f'{r_:.0%}', ha='center', color='#555555', fontsize=10)
ax1.set_title(f'Recall op nieuwbouw 2023-2024 (n={len(meetlat)})', loc='left', fontsize=10)
ax1.set_ylabel('% gevangen')
for naam, kleur in (('stedelijk', BLAUW), ('buitengebied', AMBER)):
    m = strata == naam
    ax2.scatter(pix[m], emb[m], s=4, alpha=0.25, color=kleur, label=naam, rasterized=True)
ax2.set_xlabel('pixelscore'); ax2.set_ylabel('embedding-afstand')
ax2.set_title('Samenhang per pand', loc='left', fontsize=10)
ax2.legend(frameon=False, fontsize=8, markerscale=3)
for ax in (ax1, ax2):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=0)
plt.tight_layout()
(DATA / 'mutaties_preview').mkdir(exist_ok=True)
fig.savefig(DATA / 'mutaties_preview' / 'embedding_ab.jpg', dpi=110, bbox_inches='tight')
plt.show()

## 4. Waar zijn ze het oneens?

- **Alleen embedding**: betekenis veranderde terwijl de pixels het maar matig zagen —
  verwacht: subtiele maar echte mutaties (klein paneelveld, dakkapel in schaduw).
- **Alleen pixel**: pixels veranderden maar de betekenis amper — verwacht: belichting,
  omvalling, vocht. Als dat beeld klopt, is de embedding de schonere zeef.

In [ ]:
def toon_voor_na(rijen, titel):
    if not rijen:
        print('(geen voorbeelden)'); return
    fig, assen = plt.subplots(len(rijen), 2, figsize=(9, 4.4 * len(rijen)), squeeze=False)
    for (geom, onderschrift), (as_o, as_n) in zip(rijen, assen):
        tid = tegel_id_voor(geom.centroid.x, geom.centroid.y)
        bbox = idx_oud[tid]['bbox']
        gx0, gy0, gx1, gy1 = geom.bounds
        x0, x1 = (gx0 - bbox[0]) / RES, (gx1 - bbox[0]) / RES
        y0, y1 = (bbox[3] - gy1) / RES, (bbox[3] - gy0) / RES
        m = 14 / RES
        crop = (max(0, x0 - m), max(0, y0 - m), min(PX, x1 + m), min(PX, y1 + m))
        for ax, map_, idx, titel_paneel in ((as_o, OUD_MAP, idx_oud, '2022'),
                                            (as_n, NIEUW_MAP, idx_nieuw, '2025')):
            beeld = Image.open(map_ / idx[tid]['image']).convert('RGB')
            ImageDraw.Draw(beeld).rectangle([x0, y0, x1, y1], outline=MARKEER, width=4)
            ax.imshow(beeld.crop(crop))
            ax.set_title(f'{titel_paneel} — {onderschrift}' if titel_paneel == '2022'
                         else titel_paneel, fontsize=9, loc='left')
            ax.axis('off')
    fig.suptitle(titel, fontsize=12)
    plt.tight_layout(); plt.show()

alleen_emb = sorted((resultaten[i] for i in det_emb - det_pix), key=lambda r: -r['embedding'])
alleen_pix = sorted((resultaten[i] for i in det_pix - det_emb), key=lambda r: -r['pixel'])
rijen = [(r['geom'], f"emb {r['embedding']:.3f}, pix {r['pixel']:.2f} ({r['stratum']})")
         for r in alleen_emb[:4]]
toon_voor_na(rijen, 'Gevonden door embedding, gemist door pixels')
rijen = [(r['geom'], f"pix {r['pixel']:.2f}, emb {r['embedding']:.3f} ({r['stratum']})")
         for r in alleen_pix[:4]]
toon_voor_na(rijen, 'Gevonden door pixels, niet door embedding — ruis of echt?')

## 5. Conclusie

In [ ]:
print(f'A/B op {len(resultaten)} panden, drempel P{DREMPEL_PCT} per stratum:')
print(f'- Recall meetlat: pixel {recall_pix:.0%} vs embedding {recall_emb:.0%}')
print(f'- Buitengebied-aandeel: pixel {buiten_aandeel(det_pix):.0%} '
      f'vs embedding {buiten_aandeel(det_emb):.0%}')
print(f'- Overlap {len(det_pix & det_emb)}, alleen-pixel {len(det_pix - det_emb)}, '
      f'alleen-embedding {len(det_emb - det_pix)}')
print()
print('Duiding: wint de embedding op recall bij vergelijkbare of schonere werkvoorraad,')
print('dan wordt hij de nieuwe zeef (of: combineer — detectie als één van beide aanslaat,')
print('dat maximaliseert recall en de galerijen tonen wat elk uniek bijdraagt).')
print()
print('Kanttekening: ResNet18 is getraind op gewone foto\'s, niet op luchtbeeld. Een')
print('logische verdieping is een backbone die op luchtfoto\'s is voorgetraind, of later')
print('fine-tunen met de bevestigde mutaties — dan wordt dit vanzelf het verandermodel.')

---
*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*